# T4 benchmark: TinyLlama-1.1B (fp16) on llama.cpp

Measures the numbers that `scripts/validate.py` compares against the analytical model:

| File | What |
|---|---|
| `microbench.csv` | achievable DRAM bandwidth and fp16 matmul FLOP/s (PyTorch) |
| `llama_bench.csv` | batch-1 prompt processing and generation at several context depths |
| `batched_bench.csv` | multi-sequence prefill and decode, batch 1 to 128 |
| `gpu_log.csv` | SM/memory clocks, power and temperature every 0.5 s (to spot throttling) |
| `env.json` | GPU, driver, llama.cpp commit, flags used |

**How to run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.
Expect ~25-35 min total, most of it compiling llama.cpp. The last cell downloads
`t4_llamacpp.zip`. Unzip it into `measurements/t4_llamacpp/` in the repo, then run
`python scripts/validate.py measurements/t4_llamacpp`.

In [ ]:
# --- configuration -----------------------------------------------------------
LLAMA_CPP_REF = "master"   # branch or release tag (e.g. "b6500"); the exact commit is recorded
HF_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUT = "/content/results"
SAVE_TO_DRIVE = False      # also copy results to My Drive/llm_perf_model/

N_BATCH, N_UBATCH = 2048, 512

# llama-bench (batch 1): prompt sizes, generated tokens, and context depths
BENCH_PROMPTS = [128, 512, 1024, 2048]
BENCH_GEN = 128
BENCH_DEPTHS = [0, 512, 2048, 8192]
REPS = 5

# llama-batched-bench: prompt length x parallel sequences, BB_NTG tokens generated each
BB_NPP = [128, 512, 2048]
BB_NTG = 128
BB_NPL = [1, 2, 4, 8, 16, 32, 64, 128]

import datetime, json, os, shutil, subprocess, sys
import pandas as pd

os.makedirs(OUT, exist_ok=True)
BIN = "/content/llama.cpp/build/bin"
GGUF = "/content/tinyllama-1.1b-f16.gguf"


def run(cmd, check=False, **kw):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if check and r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:], sep="\n")
        raise RuntimeError(f"command failed ({r.returncode})")
    return r

## 1. Check the GPU and start a clock/power logger

In [ ]:
q = "name,driver_version,memory.total,power.limit,clocks.max.sm,clocks.max.mem"
info = run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"], check=True).stdout.strip()
print(info)
assert "T4" in info, "Switch the runtime to a T4 GPU: Runtime > Change runtime type"
name, driver, mem, power, sm_max, mem_max = [s.strip() for s in info.split(",")]
env = dict(gpu_name=name, driver=driver, memory_total=mem, power_limit=power,
           max_sm_clock=sm_max, max_mem_clock=mem_max, hf_model=HF_MODEL,
           n_batch=N_BATCH, n_ubatch=N_UBATCH,
           timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"))

# Throttle-reason field was renamed in newer drivers; use whichever this one accepts.
fields = "timestamp,clocks.sm,clocks.mem,power.draw,temperature.gpu,utilization.gpu"
for reason in ("clocks_event_reasons.active", "clocks_throttle_reasons.active", None):
    f = fields + (f",{reason}" if reason else "")
    if run(["nvidia-smi", f"--query-gpu={f}", "--format=csv"]).returncode == 0:
        fields = f
        break
gpu_log = open(f"{OUT}/gpu_log.csv", "w")
gpu_logger = subprocess.Popen(["nvidia-smi", f"--query-gpu={fields}", "--format=csv", "-lms", "500"],
                              stdout=gpu_log)

## 2. Microbenchmarks: what the T4 actually sustains

Runs in a separate process so PyTorch releases all GPU memory before llama.cpp starts.
The T4 is a 70 W card, so sustained matmul throughput is usually well below the 65 TFLOPS datasheet number.

In [ ]:
MICRO = r"""
import time, torch
ev = lambda: torch.cuda.Event(enable_timing=True)

def timed(fn, iters):
    out = []
    for _ in range(iters):
        s, e = ev(), ev()
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        out.append(s.elapsed_time(e) / 1e3)
    return out

med = lambda xs: sorted(xs)[len(xs) // 2]
n = 512 * 2**20                                   # 1 GiB of fp16
x = torch.empty(n, dtype=torch.float16, device="cuda").uniform_()
y = torch.empty_like(x)
timed(lambda: y.copy_(x), 3)
copy_bw = 2 * n * 2 / med(timed(lambda: y.copy_(x), 20))      # read + write
timed(lambda: x.sum(dtype=torch.float32), 3)
read_bw = n * 2 / med(timed(lambda: x.sum(dtype=torch.float32), 20))
del x, y

m = 8192
a = torch.randn(m, m, dtype=torch.float16, device="cuda"); b = torch.randn_like(a)
flop = 2 * m**3
timed(lambda: a @ b, 3)
burst = flop / med(timed(lambda: a @ b, 10))
ts, t0 = [], time.time()
while time.time() - t0 < 30:                      # 30 s to reach thermal/power steady state
    ts += timed(lambda: a @ b, 10)
sustained = flop / med(ts[len(ts) // 2:])

print("metric,value,unit")
print(f"copy_bw,{copy_bw:.6g},B/s")
print(f"read_bw,{read_bw:.6g},B/s")
print(f"matmul_flops_burst,{burst:.6g},FLOP/s")
print(f"matmul_flops_sustained,{sustained:.6g},FLOP/s")
"""
r = run([sys.executable, "-c", MICRO], check=True)
open(f"{OUT}/microbench.csv", "w").write(r.stdout)
mb = pd.read_csv(f"{OUT}/microbench.csv")
mb["pct_of_peak"] = [v / p * 100 for v, p in zip(mb.value, [320e9, 320e9, 65e12, 65e12])]
mb

## 3. Build llama.cpp with CUDA (~10-20 min on Colab's 2 vCPUs)

In [ ]:
shutil.rmtree("/content/llama.cpp", ignore_errors=True)
run(["git", "clone", "--depth", "1", "--branch", LLAMA_CPP_REF,
     "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
env["llama_cpp_ref"] = LLAMA_CPP_REF
env["llama_cpp_commit"] = run(["git", "-C", "/content/llama.cpp", "rev-parse", "HEAD"],
                              check=True).stdout.strip()
print("commit", env["llama_cpp_commit"])

run(["cmake", "-S", "/content/llama.cpp", "-B", "/content/llama.cpp/build",
     "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",      # T4 = sm_75; skip other archs
     "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"], check=True)
run(["cmake", "--build", "/content/llama.cpp/build", "--config", "Release", "-j", os.cpu_count(),
     "--target", "llama-bench", "llama-batched-bench"], check=True)
assert os.path.exists(f"{BIN}/llama-bench") and os.path.exists(f"{BIN}/llama-batched-bench")

## 4. Download TinyLlama and convert to an fp16 GGUF

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(HF_MODEL, allow_patterns=["*.json", "*.safetensors", "tokenizer.model"])
if not any(f.endswith(".safetensors") for f in os.listdir(model_dir)):
    model_dir = snapshot_download(HF_MODEL, allow_patterns=["*.json", "*.bin", "tokenizer.model"])

run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)
convert = [sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", model_dir,
           "--outtype", "f16", "--outfile", GGUF]
r = run(convert)
if r.returncode != 0:   # converter needs a dependency Colab lacks; install its requirements
    print(r.stderr[-2000:])
    run([sys.executable, "-m", "pip", "install", "-q", "-r",
         "/content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt"], check=True)
    run(convert, check=True)
print(f"{os.path.getsize(GGUF) / 1e9:.2f} GB")

## 5. llama-bench: batch 1, prompt processing (pp) and generation (tg) at each context depth

Runs with flash attention both off and on. The analytical model assumes fused attention, so FA-on rows
are the main comparison, and FA-off rows show how much unfused attention costs.

In [ ]:
bench = [f"{BIN}/llama-bench", "-m", GGUF, "-ngl", 99, "-b", N_BATCH, "-ub", N_UBATCH, "-r", REPS,
         "-p", ",".join(map(str, BENCH_PROMPTS)), "-n", BENCH_GEN,
         "-d", ",".join(map(str, BENCH_DEPTHS)), "-o", "csv"]
for fa in (["-fa", "0,1"], ["-fa", "off,on"]):     # flag syntax differs across versions
    r = run(bench + fa)
    if r.returncode == 0 and r.stdout.strip():
        break
    print(r.stderr[-1500:])
else:
    raise RuntimeError("llama-bench failed")
env["llama_bench_fa_arg"] = " ".join(fa)
open(f"{OUT}/llama_bench.csv", "w").write(r.stdout)
open(f"{OUT}/llama_bench.log", "w").write(r.stderr)
lb = pd.read_csv(f"{OUT}/llama_bench.csv")
lb[["n_prompt", "n_gen", "n_depth", "flash_attn", "avg_ts", "stddev_ts"]]

## 6. llama-batched-bench: many sequences at once

For each prompt length `PP` and batch `B`, it prefills B prompts, then decodes `TG` tokens for all B sequences in lockstep.
`S_PP` and `S_TG` are prefill and decode throughput in tokens/s.

In [ ]:
def batched(npl):
    ctx = max(npl) * (max(BB_NPP) + BB_NTG)
    cmd = [f"{BIN}/llama-batched-bench", "-m", GGUF, "-ngl", 99, "-c", ctx,
           "-b", N_BATCH, "-ub", N_UBATCH, "-npp", ",".join(map(str, BB_NPP)),
           "-ntg", BB_NTG, "-npl", ",".join(map(str, npl))]
    for fa in (["-fa", "on"], ["-fa"], []):           # newer: -fa on|off|auto, older: bare -fa
        r = run(cmd + fa)
        if r.returncode == 0:
            return r, fa
        print(r.stderr[-1500:])
    return r, None

npl = list(BB_NPL)
r, fa = batched(npl)
while fa is None and len(npl) > 1:     # most likely out of memory: drop the largest batch
    npl = npl[:-1]
    print("retrying with npl =", npl)
    r, fa = batched(npl)
assert fa is not None, "llama-batched-bench failed"
env["batched_bench_fa_arg"] = " ".join(fa) or "(default)"
env["batched_bench_npl"] = npl
log = r.stdout + "\n" + r.stderr
open(f"{OUT}/batched_bench.log", "w").write(log)

rows = []
for line in log.splitlines():
    cells = [c.strip() for c in line.strip().strip("|").split("|")]
    if len(cells) == 10 and cells[0].isdigit():
        pp, tg, b, n_kv = map(int, cells[:4])
        t_pp, s_pp, t_tg, s_tg, t, s = map(float, cells[4:])
        rows.append(dict(pp=pp, tg=tg, b=b, n_kv=n_kv, t_pp=t_pp, s_pp=s_pp, t_tg=t_tg,
                         s_tg=s_tg, t=t, s=s, n_ubatch=N_UBATCH, flash_attn=fa != []))
bb = pd.DataFrame(rows)
assert len(bb), "could not parse the batched-bench table; see batched_bench.log"
bb.to_csv(f"{OUT}/batched_bench.csv", index=False)
bb

## 7. Package the results

In [ ]:
gpu_logger.terminate()
gpu_log.close()
json.dump(env, open(f"{OUT}/env.json", "w"), indent=2)

g = pd.read_csv(f"{OUT}/gpu_log.csv", skipinitialspace=True)
print(g.describe().loc[["min", "50%", "max"]].to_string())

shutil.make_archive("/content/t4_llamacpp", "zip", OUT)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/llm_perf_model", exist_ok=True)
    shutil.copy("/content/t4_llamacpp.zip", "/content/drive/MyDrive/llm_perf_model/")
from google.colab import files
files.download("/content/t4_llamacpp.zip")